# Exercise 10.1: Top-quark mass from its decay products revised

From *Programming in High Energy Particle Physics*, Chapter 10

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isildakbora/hep-programming-guide-notebooks/blob/main/ch10/ex10_1_solution.ipynb)

Exercise 10.1 Top-quark mass from its decay products revised Read a ROOT file containing simulated \(t\bar t\) events using uproot. Plot the top quark mass distribution from its decay products and compare with the known value of \(172.57 \pm 0.29\) GeV (PDG 2024 average). Expect the reconstructed peak to be much wider than the top width of about 1.4 GeV, and possibly shifted: it is broadened by the jet energy resolution and by combinatorics, since you do not know which jets come from which top quark.

<details><summary>Hint</summary>

In the lepton+jets or all-hadronic channel, reconstruct \(t\to bW\to bq\bar q'\) from three jets. Use ak.combinations(jets, 3, fields=["j1", "j2", "j3"]) , require one of them to be b-tagged, and choose the combination whose non-b dijet mass is closest to \(m_W = 80.369\) GeV, or the one with the highest three-jet \(p_T\). Compare the peak with and without this choice, and with the parton-level (truth) mass if the file has it, to see how much of the width comes from wrong combinations.

</details>

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    get_ipython().run_line_magic("pip", "install -q uproot awkward vector numpy matplotlib")

**Data:** this notebook writes a seeded toy ROOT file locally with simulated three-jet top decays and extra jets. Replace it with a documented public ttbar sample for a physical measurement.

### Step 1: Create a small simulated ROOT sample

In [ ]:
import numpy as np, awkward as ak, uproot
rng=np.random.default_rng(42); N=1500; MT=172.57; MW=80.369
p=(MT*MT-MW*MW)/(2*MT); ew=(MT*MT+MW*MW)/(2*MT)
records=[]
for _ in range(N):
    cost=rng.uniform(-1,1); phi=rng.uniform(-np.pi,np.pi); sint=np.sqrt(1-cost*cost)
    q=np.array([MW/2,MW/2*sint*np.cos(phi),MW/2*sint*np.sin(phi),MW/2*cost])
    beta=p/ew; gamma=ew/MW
    def boost(v): return np.array([gamma*(v[0]+beta*v[3]),v[1],v[2],gamma*(v[3]+beta*v[0])])
    q1=boost(q); q2=boost(np.array([q[0],-q[1],-q[2],-q[3]])); b=np.array([p,0.,0.,-p])
    vecs=[]
    for v,tag in [(q1,0),(q2,0),(b,1)]:
        v=v.copy(); v[1:]*=rng.normal(1,.09)
        pt=np.hypot(v[1],v[2]); vecs.append((pt,np.arcsinh(v[3]/max(pt,1e-5)),np.arctan2(v[2],v[1]),tag))
    for _j in range(2): vecs.append((rng.uniform(25,65),rng.normal(0,1),rng.uniform(-np.pi,np.pi),int(rng.random()<.08)))
    rng.shuffle(vecs); records.append(vecs)
arr={key:ak.Array([[v[j] for v in ev] for ev in records]) for j,key in enumerate(['jet_pt','jet_eta','jet_phi','jet_btag'])}
with uproot.recreate('toy_ttbar.root') as f: f['Events']={**arr,'top_truth_mass':np.full(N,MT)}
events=uproot.open('toy_ttbar.root')['Events'].arrays(library='ak'); assert len(events)==N

### Step 2: Form all three-jet candidates

In [ ]:
import vector
vector.register_awkward()
jets=ak.zip({'pt':events.jet_pt,'eta':events.jet_eta,'phi':events.jet_phi,
             'mass':ak.zeros_like(events.jet_pt),'btag':events.jet_btag},with_name='Momentum4D')
c=ak.combinations(jets,3,fields=['j1','j2','j3'])
raw=ak.to_numpy(ak.firsts(c.j1+c.j2+c.j3).mass)
nb=c.j1.btag+c.j2.btag+c.j3.btag
b=ak.where(c.j1.btag==1,c.j1,ak.where(c.j2.btag==1,c.j2,c.j3))
q1=ak.where(c.j1.btag==1,c.j2,c.j1)
q2=ak.where(c.j3.btag==1,c.j2,c.j3)
score=ak.where(nb==1,abs((q1+q2).mass-80.369),1e6)
best=ak.argmin(score,axis=1,keepdims=True)
chosen=ak.firsts(c[best]); selected=ak.to_numpy((chosen.j1+chosen.j2+chosen.j3).mass)
assert np.isfinite(selected).all()

### Step 3: Compare raw and W-selected top mass spectra

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(8,4)); bins=np.linspace(50,350,45)
ax.hist(raw,bins=bins,histtype='step',label='first triplet'); ax.hist(selected,bins=bins,histtype='step',label='W-selected')
ax.axvline(MT,color='black',ls='--',label='truth 172.57 GeV')
ax.set(xlabel='Three-jet invariant mass [GeV]',ylabel='Toy events / bin'); ax.legend(); plt.show()
print('Selected median [GeV]:',round(float(np.median(selected)),2),'truth [GeV]:',MT)

The selected three-jet mass is broad because of smearing and wrong jet assignments; the generator truth is fixed at 172.57 GeV in this toy. This is a reconstruction exercise, not a top-mass measurement.